# IndiGen Protein Dataset Preparation (From Existing VEP-Annotated VCF)

This notebook builds a model-ready table from your already annotated IndiGen VCF.

Target columns:
- `protein_id`
- `sequence`
- `pos`
- `ref`
- `alt`
- `label`

Notes:
- We do **not** rerun VEP.
- We parse `CSQ` from your VEP-annotated VCF.
- We keep SNV + missense logic.
- `label` is kept empty for now (you will fill it later).

In [1]:
!pip install -q cyvcf2 biopython

from google.colab import drive
drive.mount('/content/drive')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 1.6 MB/s eta 0:00:00
Mounted at /content/drive


In [2]:
import os
import re
import numpy as np
import pandas as pd
from cyvcf2 import VCF
from Bio import SeqIO

pd.set_option('display.max_columns', 200)
print('Imports ready.')

Imports ready.


## Step 1: VCF Path

We use your existing VEP-annotated IndiGen VCF (no rerun of VEP).

In [3]:
BASE = '/content/drive/MyDrive/FYP_DATA'
ANNOTATED_MANE_DIR = f'{BASE}/DATA/annotated_data_mane'
INDIGEN_VCF_PATH = f'{ANNOTATED_MANE_DIR}/indigen/indigen_annotated/indigen_annotated_mane.vcf.gz'

path_df = pd.DataFrame([
    {
        'resource': 'indigen_annotated_vcf',
        'path': INDIGEN_VCF_PATH,
        'exists': os.path.exists(INDIGEN_VCF_PATH)
    }
])

display(path_df)

,resource,path,exists
0,indigen_annotated_vcf,/content/drive/MyDrive/FYP_DATA/DATA/annotated...,True


## Step 2: Extract Missense Protein Changes From CSQ

What this does:
- Reads VCF + CSQ
- Keeps SNVs
- Keeps entries where consequence includes `missense_variant`
- Extracts `protein_id`, `pos`, `ref`, `alt` at amino-acid level

In [8]:
def get_csq_fields(vcf):
    header = vcf.raw_header
    for line in header.split('\n'):
        if 'ID=CSQ' in line and 'Format: ' in line:
            fmt_start = line.index('Format: ') + len('Format: ')
            fmt_end = line.index('\">')
            return line[fmt_start:fmt_end].split('|')
    return None


def is_snv(ref, alt):
    return len(ref) == 1 and len(alt) == 1


def safe_idx(csq_fields, name):
    return csq_fields.index(name) if (csq_fields and name in csq_fields) else None


def parse_hgvsp_protein_id(hgvsp_value):
    if hgvsp_value is None:
        return None
    txt = str(hgvsp_value)
    if ':p.' in txt:
        left = txt.split(':p.')[0]
        if left.startswith('ENSP'):
            return left.split('.')[0]
    return None


def parse_protein_position(pos_text):
    if pos_text is None:
        return None
    txt = str(pos_text).strip()
    if txt in ['', '-', 'None', 'nan']:
        return None
    # Accept: 197, 197-198, 197/401, 197-198/401
    left = txt.split('/')[0]
    first = left.split('-')[0]
    return int(first) if first.isdigit() else None


def parse_amino_acids(aa_text):
    if aa_text is None:
        return (None, None)
    txt = str(aa_text)
    if '/' not in txt:
        return (None, None)
    left, right = txt.split('/', 1)
    if len(left) == 1 and len(right) == 1:
        return (left, right)
    return (None, None)


def extract_protein_level_missense(vcf_path):
    # Keep transcript_id as fallback because some VEP VCFs do not emit ENSP in CSQ.
    empty = pd.DataFrame(columns=['transcript_id', 'protein_id', 'pos', 'ref', 'alt'])

    if (vcf_path is None) or (not os.path.exists(vcf_path)):
        return {
            'status': 'missing',
            'total_variants': 0,
            'snv_variants': 0,
            'missense_tx_hits': 0,
            'protein_df': empty,
        }

    vcf = VCF(vcf_path)
    csq_fields = get_csq_fields(vcf)
    consequence_idx = safe_idx(csq_fields, 'Consequence')
    hgvsp_idx = safe_idx(csq_fields, 'HGVSp')
    feature_idx = safe_idx(csq_fields, 'Feature')
    feature_type_idx = safe_idx(csq_fields, 'Feature_type')
    protein_pos_idx = safe_idx(csq_fields, 'Protein_position')
    amino_acids_idx = safe_idx(csq_fields, 'Amino_acids')

    total_variants = 0
    snv_variants = 0
    missense_tx_hits = 0
    rows = []

    for var in vcf:
        total_variants += 1

        ref_nt = var.REF
        alt_nt = var.ALT[0] if var.ALT else ''
        if not is_snv(ref_nt, alt_nt):
            continue

        snv_variants += 1
        csq = var.INFO.get('CSQ')
        if (csq is None) or (consequence_idx is None):
            continue

        for tx in str(csq).split(','):
            vals = tx.split('|')
            consequence = vals[consequence_idx] if consequence_idx < len(vals) else ''
            if 'missense_variant' not in consequence.split('&'):
                continue

            missense_tx_hits += 1

            hgvsp = vals[hgvsp_idx] if (hgvsp_idx is not None and hgvsp_idx < len(vals)) else ''
            feature = vals[feature_idx] if (feature_idx is not None and feature_idx < len(vals)) else ''
            feature_type = vals[feature_type_idx] if (feature_type_idx is not None and feature_type_idx < len(vals)) else ''
            protein_pos = vals[protein_pos_idx] if (protein_pos_idx is not None and protein_pos_idx < len(vals)) else ''
            amino_acids = vals[amino_acids_idx] if (amino_acids_idx is not None and amino_acids_idx < len(vals)) else ''

            transcript_id = None
            if feature_type == 'Transcript' and str(feature).startswith('ENST'):
                transcript_id = str(feature).split('.')[0]

            protein_id = parse_hgvsp_protein_id(hgvsp)
            pos = parse_protein_position(protein_pos)
            aa_ref, aa_alt = parse_amino_acids(amino_acids)

            if transcript_id and (pos is not None) and aa_ref and aa_alt:
                rows.append({
                    'transcript_id': transcript_id,
                    'protein_id': protein_id,  # may be None; will fill from FASTA mapping
                    'pos': pos,
                    'ref': aa_ref,
                    'alt': aa_alt,
                })

    protein_df = pd.DataFrame(rows).drop_duplicates().reset_index(drop=True)

    return {
        'status': 'ok',
        'total_variants': total_variants,
        'snv_variants': snv_variants,
        'missense_tx_hits': missense_tx_hits,
        'protein_df': protein_df,
    }


protein_extract_result = extract_protein_level_missense(INDIGEN_VCF_PATH)
print('Extraction status:', protein_extract_result['status'])
print('Total variants:', f"{protein_extract_result['total_variants']:,}")
print('SNV variants:', f"{protein_extract_result['snv_variants']:,}")
print('Missense transcript hits:', f"{protein_extract_result['missense_tx_hits']:,}")
print('Unique transcript/protein-level rows:', f"{len(protein_extract_result['protein_df']):,}")
if not protein_extract_result['protein_df'].empty:
    print('Rows with direct protein_id from CSQ:', protein_extract_result['protein_df']['protein_id'].notna().sum())

display(protein_extract_result['protein_df'].head(20))

Extraction status: ok
Total variants: 16,266,617
SNV variants: 14,503,055
Missense transcript hits: 807,415
Unique transcript/protein-level rows: 807,386
Rows with direct protein_id from CSQ: 0


,transcript_id,protein_id,pos,ref,alt
0,ENST00000641515,None,164,G,V
1,ENST00000616016,None,48,P,A
2,ENST00000618323,None,48,P,A
3,ENST00000968542,None,48,P,A
4,ENST00000968543,None,48,P,A
5,ENST00000968544,None,48,P,A
6,ENST00000616016,None,80,A,S
7,ENST00000618323,None,80,A,S
8,ENST00000968542,None,80,A,S
9,ENST00000968543,None,80,A,S


# DEBUGGING

In [7]:
# Debugging: inspect missense transcript fields and test robust parsing

def parse_protein_position_flexible(pos_text):
    if pos_text is None:
        return None
    txt = str(pos_text).strip()
    if txt in ['', '-', 'None', 'nan']:
        return None

    # Common VEP patterns handled:
    # 197
    # 197/401
    # 197-198/401
    # 197-198
    left = txt.split('/')[0]
    first = left.split('-')[0]
    return int(first) if first.isdigit() else None


def debug_missense_fields(vcf_path, sample_n=30):
    vcf = VCF(vcf_path)
    csq_fields = get_csq_fields(vcf)

    consequence_idx = safe_idx(csq_fields, 'Consequence')
    hgvsp_idx = safe_idx(csq_fields, 'HGVSp')
    feature_idx = safe_idx(csq_fields, 'Feature')
    feature_type_idx = safe_idx(csq_fields, 'Feature_type')
    protein_pos_idx = safe_idx(csq_fields, 'Protein_position')
    amino_acids_idx = safe_idx(csq_fields, 'Amino_acids')

    dbg_rows = []
    scanned = 0

    for var in vcf:
        scanned += 1
        csq = var.INFO.get('CSQ')
        if (csq is None) or (consequence_idx is None):
            continue

        for tx in str(csq).split(','):
            vals = tx.split('|')
            consequence = vals[consequence_idx] if consequence_idx < len(vals) else ''
            if 'missense_variant' not in consequence.split('&'):
                continue

            hgvsp = vals[hgvsp_idx] if (hgvsp_idx is not None and hgvsp_idx < len(vals)) else ''
            feature = vals[feature_idx] if (feature_idx is not None and feature_idx < len(vals)) else ''
            feature_type = vals[feature_type_idx] if (feature_type_idx is not None and feature_type_idx < len(vals)) else ''
            protein_pos_raw = vals[protein_pos_idx] if (protein_pos_idx is not None and protein_pos_idx < len(vals)) else ''
            amino_acids_raw = vals[amino_acids_idx] if (amino_acids_idx is not None and amino_acids_idx < len(vals)) else ''

            parsed_protein_id = parse_hgvsp_protein_id(hgvsp)
            if parsed_protein_id is None and feature_type == 'Transcript' and str(feature).startswith('ENSP'):
                parsed_protein_id = str(feature).split('.')[0]

            parsed_pos_old = parse_protein_position(protein_pos_raw)
            parsed_pos_new = parse_protein_position_flexible(protein_pos_raw)
            aa_ref, aa_alt = parse_amino_acids(amino_acids_raw)

            dbg_rows.append({
                'CHROM': var.CHROM,
                'POS': var.POS,
                'HGVSp': hgvsp,
                'Feature_type': feature_type,
                'Feature': feature,
                'Protein_position_raw': protein_pos_raw,
                'Amino_acids_raw': amino_acids_raw,
                'protein_id_parsed': parsed_protein_id,
                'pos_old_parser': parsed_pos_old,
                'pos_new_parser': parsed_pos_new,
                'aa_ref': aa_ref,
                'aa_alt': aa_alt,
            })

            if len(dbg_rows) >= sample_n:
                dbg_df = pd.DataFrame(dbg_rows)
                print(f'Scanned variants: {scanned:,}')
                print(f'Debug rows collected: {len(dbg_df):,}')
                return dbg_df

    dbg_df = pd.DataFrame(dbg_rows)
    print(f'Scanned variants: {scanned:,}')
    print(f'Debug rows collected: {len(dbg_df):,}')
    return dbg_df


debug_df = debug_missense_fields(INDIGEN_VCF_PATH, sample_n=30)
display(debug_df)

if not debug_df.empty:
    print('\nOld parser success count:', debug_df['pos_old_parser'].notna().sum())
    print('New parser success count:', debug_df['pos_new_parser'].notna().sum())
    print('Protein ID parsed count:', debug_df['protein_id_parsed'].notna().sum())
    print('AA pair parsed count:', (debug_df['aa_ref'].notna() & debug_df['aa_alt'].notna()).sum())

Scanned variants: 1,709
Debug rows collected: 30


,CHROM,POS,HGVSp,Feature_type,Feature,Protein_position_raw,Amino_acids_raw,protein_id_parsed,pos_old_parser,pos_new_parser,aa_ref,aa_alt
0,chr1,69518,,Transcript,ENST00000641515,164,G/V,None,164,164,G,V
1,chr1,924573,,Transcript,ENST00000616016,48,P/A,None,48,48,P,A
2,chr1,924573,,Transcript,ENST00000618323,48,P/A,None,48,48,P,A
3,chr1,924573,,Transcript,ENST00000968542,48,P/A,None,48,48,P,A
4,chr1,924573,,Transcript,ENST00000968543,48,P/A,None,48,48,P,A
5,chr1,924573,,Transcript,ENST00000968544,48,P/A,None,48,48,P,A
6,chr1,924669,,Transcript,ENST00000616016,80,A/S,None,80,80,A,S
7,chr1,924669,,Transcript,ENST00000618323,80,A/S,None,80,80,A,S
8,chr1,924669,,Transcript,ENST00000968542,80,A/S,None,80,80,A,S
9,chr1,924669,,Transcript,ENST00000968543,80,A/S,None,80,80,A,S



Old parser success count: 30
New parser success count: 30
Protein ID parsed count: 0
AA pair parsed count: 30


# -------------------- END DEBUGGING ------------------------

## Step 3: Confirm FASTA Path (Required)

Stop here and set your reference proteome FASTA path in the next cell.

Expected file example:
- `Homo_sapiens.GRCh38.pep.all.fa`

In [10]:
# Confirmed FASTA path
FASTA_PATH = '/content/drive/MyDrive/FYP_DATA/DATA/raw_data/reference_fasta/Homo_sapiens.GRCh38.pep.all.fa'

print('FASTA path set to:', FASTA_PATH)
print('Exists:', os.path.exists(FASTA_PATH))

FASTA path set to: /content/drive/MyDrive/FYP_DATA/DATA/raw_data/reference_fasta/Homo_sapiens.GRCh38.pep.all.fa
Exists: True


## Step 4: Attach Protein Sequences

This step loads the FASTA proteome and attaches `sequence` to each `protein_id`.

In [11]:
def load_proteome_and_transcript_map(fasta_path):
    proteome = {}
    tx_to_pid = {}

    for record in SeqIO.parse(fasta_path, 'fasta'):
        pid = str(record.id).split('.')[0]  # ENSP base id
        proteome[pid] = str(record.seq)

        # Ensembl FASTA headers usually contain transcript:ENST...
        m = re.search(r'transcript:(ENST\d+)', str(record.description))
        if m:
            tx_id = m.group(1)
            if tx_id not in tx_to_pid:
                tx_to_pid[tx_id] = pid

    return proteome, tx_to_pid


protein_df = protein_extract_result['protein_df'].copy()
proteome, tx_to_pid = load_proteome_and_transcript_map(FASTA_PATH)
print('Loaded protein sequences:', f"{len(proteome):,}")
print('Transcript->Protein map size:', f"{len(tx_to_pid):,}")

# Fill missing protein_id using transcript_id from FASTA transcript mapping
if not protein_df.empty:
    missing_before = protein_df['protein_id'].isna().sum()
    protein_df.loc[protein_df['protein_id'].isna(), 'protein_id'] = (
        protein_df.loc[protein_df['protein_id'].isna(), 'transcript_id'].map(tx_to_pid)
    )
    missing_after = protein_df['protein_id'].isna().sum()
    print('Missing protein_id before fill:', f"{missing_before:,}")
    print('Missing protein_id after fill:', f"{missing_after:,}")

protein_df['sequence'] = protein_df['protein_id'].map(proteome)
protein_df['label'] = pd.NA

# Keep requested final schema
final_df = protein_df[['protein_id', 'sequence', 'pos', 'ref', 'alt', 'label']].copy()

# QC: verify reference amino acid matches sequence at position
valid_mask = (
    final_df['sequence'].notna()
    & final_df['pos'].notna()
    & (final_df['pos'] > 0)
)

def aa_match(row):
    seq = row['sequence']
    pos = int(row['pos'])
    ref = row['ref']
    if (seq is None) or (pos <= 0) or (pos > len(seq)):
        return False
    return seq[pos - 1] == ref

final_df['ref_match'] = final_df.apply(lambda r: aa_match(r) if valid_mask.loc[r.name] else False, axis=1)

model_ready_df = final_df[final_df['ref_match']].drop(columns=['ref_match']).reset_index(drop=True)
qc_failed_df = final_df[~final_df['ref_match']].reset_index(drop=True)

print('Rows after sequence mapping:', f"{len(final_df):,}")
print('Rows passing ref-aa QC:', f"{len(model_ready_df):,}")
print('Rows failing QC:', f"{len(qc_failed_df):,}")

display(model_ready_df.head(20))

Loaded protein sequences: 245,535
Transcript->Protein map size: 245,535
Missing protein_id before fill: 807,386
Missing protein_id after fill: 0
Rows after sequence mapping: 807,386
Rows passing ref-aa QC: 807,383
Rows failing QC: 3


,protein_id,sequence,pos,ref,alt,label
0,ENSP00000493376,MKKVTAEAISWNESTSETNNSMVTEFIFLGLSDSQELQTFLFMLFF...,164,G,V,<NA>
1,ENSP00000478421,MPAVKKEFPGREDLALALATFHPTLAALPLPPLPGYLAPLPAAAAL...,48,P,A,<NA>
2,ENSP00000480678,MPAVKKEFPGREDLALALATFHPTLAALPLPPLPGYLAPLPAAAAL...,48,P,A,<NA>
3,ENSP00000638601,MPAVKKEFPGREDLALALATFHPTLAALPLPPLPGYLAPLPAAAAL...,48,P,A,<NA>
4,ENSP00000638602,MPAVKKEFPGREDLALALATFHPTLAALPLPPLPGYLAPLPAAAAL...,48,P,A,<NA>
5,ENSP00000638603,MPAVKKEFPGREDLALALATFHPTLAALPLPPLPGYLAPLPAAAAL...,48,P,A,<NA>
6,ENSP00000478421,MPAVKKEFPGREDLALALATFHPTLAALPLPPLPGYLAPLPAAAAL...,80,A,S,<NA>
7,ENSP00000480678,MPAVKKEFPGREDLALALATFHPTLAALPLPPLPGYLAPLPAAAAL...,80,A,S,<NA>
8,ENSP00000638601,MPAVKKEFPGREDLALALATFHPTLAALPLPPLPGYLAPLPAAAAL...,80,A,S,<NA>
9,ENSP00000638602,MPAVKKEFPGREDLALALATFHPTLAALPLPPLPGYLAPLPAAAAL...,80,A,S,<NA>


## Step 5: Export Outputs

- `model_ready_df`: final dataset (`protein_id, sequence, pos, ref, alt, label`)
- `qc_failed_df`: rows failing sequence/ref quality checks

In [12]:
SAVE_OUTPUTS = True
OUTPUT_DIR = f'{BASE}/DATA/model_ready_outputs'

if SAVE_OUTPUTS:
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    final_out = f'{OUTPUT_DIR}/indigen_model_ready_unlabeled.csv'
    failed_out = f'{OUTPUT_DIR}/indigen_qc_failed_rows.csv'

    model_ready_df.to_csv(final_out, index=False)
    qc_failed_df.to_csv(failed_out, index=False)

    print('Saved:', final_out)
    print('Saved:', failed_out)
else:
    print('SAVE_OUTPUTS is False. No files were written.')

Saved: /content/drive/MyDrive/FYP_DATA/DATA/model_ready_outputs/indigen_model_ready_unlabeled.csv
Saved: /content/drive/MyDrive/FYP_DATA/DATA/model_ready_outputs/indigen_qc_failed_rows.csv
